# Data Ingestion
- Data is stored in supabase

## Extracting files from Supabase Storage (boto3)

### Importing libraries ans system variables

In [1]:
import boto3
import pandas as pd
import io
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
# importing variables
load_dotenv()
s3_endpoint = os.getenv("S3_ENDPOINT_URL")
aws_region = os.getenv("AWS_REGION")
aws_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret = os.getenv("AWS_SECRET_ACCESS_KEY")
bucket = os.getenv("BUCKET_NAME")
database_url = os.getenv("DATABASE_URL")

### Creating Boto3 client connection

In [3]:
# Creating connection
s3 = boto3.client(
    "s3",
    region_name = aws_region,
    endpoint_url = s3_endpoint,
    aws_access_key_id = aws_key,
    aws_secret_access_key = aws_secret
)

### Verify response

In [4]:
# getting the payload
response_names = s3.list_objects_v2(Bucket= bucket)
display(response_names)

{'ResponseMetadata': {'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 13 May 2026 14:46:13 GMT',
   'content-type': 'text/plain',
   'content-length': '1304',
   'connection': 'keep-alive',
   'cf-ray': '9fb2738e490196b3-DUB',
   'cf-cache-status': 'DYNAMIC',
   'access-control-allow-origin': '*',
   'server': 'cloudflare',
   'strict-transport-security': 'max-age=31536000; includeSubDomains; preload',
   'content-security-policy': "default-src 'none'; sandbox",
   'x-content-type-options': 'nosniff',
   'sb-gateway-mode': 'direct',
   'sb-gateway-version': '1',
   'sb-project-ref': 'bkwfxsvbmvcbuucoytzq',
   'sb-request-id': '019e21cd-74f2-7e11-9cdb-c034b677c9fc',
   'set-cookie': '__cf_bm=D2c8ISG..KBmhe1AN29cns069tO2NgLAMzUJkvlpDcM-1778683573.483585-1.0.1.1-hqtwk8G7fkkZp1UdTj_VpWlo6i3ZSJFK.7Ri8HXdOcZjwiquYADmJlV_oYyfF.DuhCbM6P4bizBE1Lqvs7.50Q3beriIvVG2XdRirJ1BON1zQf0zIGMQhJgWb9w03nKM; HttpOnly; SameSite=None; Secure; Path=/; Domain=storage.supabase.co; Expires=Wed, 13 May 2026

In [5]:
# Storing files to read in Pandas
files = [obj["Key"] for obj in response_names["Contents"]]
print(files)

['.emptyFolderPlaceholder', 'competitor_prices.parquet', 'customers.parquet', 'products.parquet', 'sales.parquet']


In [6]:
# Checking if the files are being read
file_key = files[1]
response_objects = s3.get_object(Bucket= bucket, Key=file_key)
parquet_bytes = response_objects["Body"].read()
print(parquet_bytes)

b'PAR1\x15\x00\x15\xce\xe3\x01\x15\x929,\x15\xb0\x0b\x15\x00\x15\x06\x15\x06\x00\x00\xe7qh\x03\x00\x00\x00\xb0\x0b\x01\x10\x00\x00\x00prd_2293732b7542\xbe\x14\x00,84a009f1889d\x11<\xee\x14\x00,ff35b88df534\x11P\xee\x14\x00,38c1b898aecc\x11P\xee\x14\x00,4ee65ed59110\x11P,dfac11815029\xfe\x14\x00\x01\x14(334f4a32f97\x15P\xee\x14\x00,b7490508211e\x11\xa0\xee\x14\x00(23664e42a715T\x9e\x14\x00,045813f96891\x11\x8cN\x14\x00(6d4e3b1efd0\x15d\xee\x14\x00$afd58e5fd0Y\xa8\xee\x14\x00(686e2aab873U\xf8\xee\x14\x00,a4025bac91a71\x18\x9e\x14\x00(bf5d8671f7aU0\x9e\x14\x00(205f309c1a25\x90\xee\x14\x00,92c0ecf60a58\x11\xc8\x9e\x14\x00(385035a5bc2u\\\x9e\x14\x00(9239eec89825\x90\xee\x14\x00(8b22fd85d50\x15\x8c\x9e\x14\x00\x014\x18fa66507up\xf2\x14\x00$92ad43fa13U\x1c\x9e\x14\x00$645f7859f3\x99L\xee\x14\x00 82cd17cb5.\xf0\x05\xea\x14\x00(f063f46a7efU0N\x14\x00\x1cd22d1b8b\xa1eQXN\x14\x00(30518d908a9\x15(\xee\x14\x00$c62100cc28\x99\xec\xee\x14\x00(2cbafff50c7\x15\xa0\xee\x14\x00,51511e44c92a1\x18\xee\x14\

### Read Table customer as Tests

In [7]:
# Using pandas and IO to read the files
df_customer = pd.read_parquet(io.BytesIO(parquet_bytes))
display(df_customer)

,id_produto,nome_concorrente,preco_concorrente,data_coleta
0,prd_2293732b7542,Mercado Livre,65.45,2026-01-11 17:35:52
1,prd_2293732b7542,Amazon,67.52,2026-01-11 17:11:42
2,prd_2293732b7542,Shopee,70.97,2026-01-11 17:45:49
3,prd_84a009f1889d,Mercado Livre,31.25,2026-01-11 18:27:37
4,prd_84a009f1889d,Amazon,31.25,2026-01-11 16:50:51
...,...,...,...,...
723,prd_00f202452062,Magalu,167.48,2026-01-11 20:21:00
724,prd_00f202452062,Shopee,176.03,2026-01-11 22:40:42
725,prd_5da705f2fce5,Mercado Livre,48.02,2026-01-11 17:31:26
726,prd_5da705f2fce5,Magalu,49.00,2026-01-11 18:15:09


## Loading Tables to Supabase SQL

In [8]:
# Creating engine to use SQL Alchemy
engine = create_engine(database_url)


In [13]:
# Ingesting tables to Supabase
print("\n" + "=" * 50)
print("Ingesting tables to Supabase...")
print("=" * 50 + "\n")

# Table list 
TABLES = ["customers","products","sales","competitor_prices"]

# Dictionary to store the dataframes
dataframes = {}

# Load tables into 
for table in TABLES:
    print(f"Downloading table {table}.parquet from Datalake...")

    # Assemble file name: "products.parquet"
    file_key = f"{table}.parquet"

    # Download file from storage
    response_objects = s3.get_object(Bucket=bucket, Key=file_key)
    parquet_bytes = response_objects["Body"].read()

    #Convert bytes to dataframe
    dataframes[table] = pd.read_parquet(io.BytesIO(parquet_bytes))

    print(f"{table}: {len(dataframes[table])} rows loaded")

for table_name, df in dataframes.items():
    print("\n")
    print(f" Saving {table_name} into PostgesSQL...")

    # Loading tables
    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )

    print(f"{table_name}: {len(dataframes[table_name])} loaded to the database")

print("\n")
print("Final Checks")

for check_table in TABLES:
    df_check = pd.read_sql_query(
        f"Select count(*) as total from {check_table}",
        engine
    )

    total = df_check["total"].iloc[0]

    print(f"{check_table}:{total} rows into database")


Ingesting tables to Supabase...

customers: 50 rows loaded
products: 215 rows loaded
sales: 3020 rows loaded
competitor_prices: 728 rows loaded


 Saving customers into PostgesSQL...
customers: 50 loaded to the database


 Saving products into PostgesSQL...
products: 215 loaded to the database


 Saving sales into PostgesSQL...
sales: 3020 loaded to the database


 Saving competitor_prices into PostgesSQL...
competitor_prices: 728 loaded to the database


Final Checks
customers:50 rows into database
products:215 rows into database
sales:3020 rows into database
competitor_prices:728 rows into database
